# SPV-MIA — Milestone 2 (Nicole Shantz)
**Method:** Self-calibrated Probabilistic Variation Membership Inference Attack  
**Based on:** Fu et al., NeurIPS 2024  
**Dataset:** UBC-SLIME/colx_531_group_project (clinical discharge notes)  
**Target model:** SmolLM2-360M fine-tuned on discharge notes

---

### What this notebook does
Implements SPV-MIA with three membership signals:
1. **PDC** — `ref_loss − target_loss` using a self-prompted reference model
2. **Grey-box** — mean L2 norm of last hidden state (free, same forward pass)
3. **PVA** — probability curvature via T5 symmetric paraphrasing (disabled by default — takes ~27hrs)

### How to use
1. Set `SCORE_SPLIT = "validation"` → run all cells → get AUC + TPR for your report table  
2. Set `SCORE_SPLIT = "test"` → re-run Step 9 only (skip Steps 5–6, ref model is saved) → upload CSV to Kaggle


## Step 1: Install Dependencies

In [1]:
# Install order matters on Colab T4 — unsloth before transformers
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q 2>/dev/null || \
    echo "Unsloth unavailable — falling back to standard HF loader"

!pip install transformers datasets scikit-learn torch tqdm pandas sentencepiece -q

# Flash Attention 2 — optional, T4 supports it but compile takes ~3-5 min
!pip install flash-attn --no-build-isolation -q 2>/dev/null || \
    echo "Flash Attention 2 unavailable — using standard attention"

print("✓ Dependencies ready")

Flash Attention 2 unavailable — using standard attention
✓ Dependencies ready


## Step 2: Imports & Device Setup

In [2]:
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"  

import os, random, json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
from datasets import load_dataset, Dataset as HFDataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments,
)
from sklearn.metrics import roc_auc_score, roc_curve

# Unsloth — fast LLaMA-family loading (SmolLM2 is LLaMA-based)
try:
    from unsloth import FastLanguageModel
    UNSLOTH_AVAILABLE = True
    print("✓ Unsloth available")
except ImportError:
    UNSLOTH_AVAILABLE = False
    print("  Unsloth not found — using standard HF loader")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# float16
if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    _ampere  = any(x in gpu_name for x in ["A100","A10","L4","H100","A6000","3090","4090"])
    DTYPE    = torch.bfloat16 if _ampere else torch.float16
    print(f"  GPU  : {gpu_name}")
    print(f"  Dtype: {'bfloat16 (Ampere+ native)' if DTYPE == torch.bfloat16 else 'float16 (T4 tensor cores)'}")
else:
    DTYPE = torch.float32

FLASH_ATTN = False
if DEVICE == "cuda":
    try:
        import flash_attn; FLASH_ATTN = True; print("✓ Flash Attention 2 available")
    except ImportError:
        print("  Flash Attention 2 not available — using standard attention")

print(f"\nDevice={DEVICE}  dtype={DTYPE}  FlashAttn2={FLASH_ATTN}")

/home/nicole-shantz/miniforge3/envs/colx_531/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_332567/2099426384.py:19: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
  Unsloth not found — using standard HF loader
  GPU  : NVIDIA GeForce RTX 2080 Super with Max-Q Design
  Dtype: float16 (T4 tensor cores)
  Flash Attention 2 not available — using standard attention

Device=cuda  dtype=torch.float16  FlashAttn2=False


## Step 3: Configuration

In [ ]:
# Models 
TARGET_MODEL   = "UBC-SLIME/colx_531_smollm2-360m"
REF_BASE_MODEL = "HuggingFaceTB/SmolLM2-360M"
REF_MODEL_DIR  = "./spv_ref_model"

# Dataset 
DATASET_NAME = "UBC-SLIME/colx_531_group_project"
SCORE_SPLIT  = "validation"  #"validation" to get AUC; "test" for Kaggle

# Self-prompt generation 
N_GENERATE  = 1000  
PROMPT_LEN  = 32
REF_EPOCHS  = 2     
REF_BATCH   = 8
SCORE_BATCH_SIZE = 32
MAX_LENGTH  = 512

## Batched scoring 

# Signals
USE_GREYBOX = False  
USE_PVA     = False  

# PVA (only used if USE_PVA=True)
PVA_N_PAIRS    = 5
PVA_MASK_RATIO = 0.2

# Output 
OUTPUT_CSV = "submission_spv_mia.csv"
CKPT_DIR   = "./spv_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

print("Configuration:")
print(f"  SCORE_SPLIT      = {SCORE_SPLIT}")
print(f"  N_GENERATE       = {N_GENERATE}")
print(f"  REF_EPOCHS       = {REF_EPOCHS}")
print(f"  SCORE_BATCH_SIZE = {SCORE_BATCH_SIZE}")
print(f"  USE_GREYBOX      = {USE_GREYBOX}  ← free signal")
print(f"  USE_PVA          = {USE_PVA}  ← disabled (27hrs otherwise)")

Configuration:
  SCORE_SPLIT      = validation
  N_GENERATE       = 1000
  REF_EPOCHS       = 2
  SCORE_BATCH_SIZE = 32
  USE_GREYBOX      = True  ← free signal
  USE_PVA          = False  ← disabled (27hrs otherwise)


## Step 4: Model Loader

Priority: Unsloth (2× faster, ~40% less VRAM) → HF + Flash Attn2 → HF standard.  
`torch.compile()` wrapped in try/except — T4 Triton JIT sometimes fails in Colab.

In [5]:
def load_causal_model(name, for_inference=True):
    """Load causal LM with best available optimizations."""
    print(f"  Loading {name} ...")
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"   # left-pad: all tokens right-aligned in batches

    if UNSLOTH_AVAILABLE and DEVICE == "cuda":
        model, _ = FastLanguageModel.from_pretrained(
            model_name=name, dtype=DTYPE, load_in_4bit=False)
        if for_inference:
            FastLanguageModel.for_inference(model)
        model = model.to(DEVICE)
    else:
        attn_kwargs = {"attn_implementation": "flash_attention_2"} if FLASH_ATTN else {}
        model = AutoModelForCausalLM.from_pretrained(
            name, torch_dtype=DTYPE, **attn_kwargs).to(DEVICE)

    if for_inference:
        model.eval()
        print("    torch.compile() disabled — avoids Unsloth patch conflicts")

    loader = "Unsloth" if (UNSLOTH_AVAILABLE and DEVICE == "cuda") else "HF"
    print(f"  ✓ Loaded  ({DTYPE}, {loader})")
    return tok, model

target_tok, target_model = load_causal_model(TARGET_MODEL, for_inference=True)
print("\n✓ Target model ready")

  Loading UBC-SLIME/colx_531_smollm2-360m ...


[huggingface_hub.utils._http|WARNING]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 677.81it/s, Materializing param=model.norm.weight]                              


    torch.compile() disabled — avoids Unsloth patch conflicts
  ✓ Loaded  (torch.float16, HF)

✓ Target model ready


## Step 5: Generate Self-Prompt Reference Dataset (PDC module)

**Why:** Standard LiRA uses the unfine-tuned base model as reference — but a base model that knows nothing about the target domain gives a weak calibration signal.  
SPV-MIA instead prompts the target model with short prefixes and collects its own generated continuations (~1,000 texts). These approximate the private training distribution without accessing that data directly.  
Fu et al. show this approach reaches within 3% of AUC of an identical-distribution reference dataset, even from unrelated seed prompts (§5.3, Fig. 3).

In [6]:
def generate_self_prompt_texts(target_tok, target_model, seed_texts,
                                n_generate, prompt_len=32, max_new_tokens=128,
                                batch_size=8):
    print(f"Generating {n_generate} self-prompt texts ...")
    generated, seeds = [], random.choices(seed_texts, k=n_generate)
    for i in tqdm(range(0, n_generate, batch_size)):
        batch = seeds[i : i + batch_size]
        prompts = [target_tok.decode(
            target_tok(s, truncation=True, max_length=prompt_len)["input_ids"],
            skip_special_tokens=True) for s in batch]
        inputs = target_tok(
            prompts, return_tensors="pt",
            truncation=True, max_length=prompt_len, padding=True).to(DEVICE)
        with torch.no_grad():   # no gradients needed for generation
            out = target_model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=True, temperature=0.9, top_p=0.95,
                pad_token_id=target_tok.eos_token_id)
        for seq in out:
            generated.append(target_tok.decode(seq, skip_special_tokens=True))
    return generated[:n_generate]

print("Loading seed texts from train split ...")
train_ds   = load_dataset(DATASET_NAME, split="train")
seed_texts = list(train_ds["text"])[:5000]

generated_texts = generate_self_prompt_texts(
    target_tok, target_model, seed_texts, N_GENERATE, PROMPT_LEN)

with open(f"{CKPT_DIR}/generated_texts.txt", "w") as f:
    f.write("\n---\n".join(generated_texts))
print(f"\n✓ {len(generated_texts)} texts saved → {CKPT_DIR}/generated_texts.txt")

Loading seed texts from train split ...
Generating 1000 self-prompt texts ...


100%|██████████| 125/125 [33:50<00:00, 16.24s/it]


✓ 1000 texts saved → ./spv_checkpoints/generated_texts.txt


## Step 6: Fine-tune Reference Model on Self-Generated Corpus

The unfine-tuned SmolLM2-360M base is fine-tuned for 2 epochs on the generated texts. This teaches it the same distribution the target model was trained on, making it a much better calibrator for the PDC signal.

In [7]:
# Step 6 — save base model as reference (skip fine-tuning; Unsloth patches
# make training impossible in the same session as inference)
print("Saving base model as reference...")
tok_tmp = AutoTokenizer.from_pretrained(REF_BASE_MODEL)
if tok_tmp.pad_token is None:
    tok_tmp.pad_token = tok_tmp.eos_token
model_tmp = AutoModelForCausalLM.from_pretrained(REF_BASE_MODEL, torch_dtype=torch.float32)
os.makedirs(REF_MODEL_DIR, exist_ok=True)
model_tmp.save_pretrained(REF_MODEL_DIR)
tok_tmp.save_pretrained(REF_MODEL_DIR)
del model_tmp
print(f"\u2713 Base model saved as reference \u2192 {REF_MODEL_DIR}")

Saving base model as reference...


Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.81s/it]


✓ Base model saved as reference → ./spv_ref_model


## Step 7: Load Fine-tuned Reference Model

In [8]:
ref_tok, ref_model = load_causal_model(REF_MODEL_DIR, for_inference=True)
print("✓ Reference model ready")

  Loading ./spv_ref_model ...


Loading weights: 100%|██████████| 290/290 [00:02<00:00, 141.65it/s, Materializing param=model.norm.weight]                              


    torch.compile() disabled — avoids Unsloth patch conflicts
  ✓ Loaded  (torch.float16, HF)
✓ Reference model ready


## Step 8: PVA Paraphraser — T5 Semantic Domain (optional)

Only instantiated when `USE_PVA=True`. Disabled by default due to ~27hr runtime on 15k samples.

**How it works (Algorithm 2, Fu et al.):**
1. Randomly mask 20% of tokens in the text  
2. T5-base predicts the masked tokens → x⁺  
3. Find x⁻ by mirroring x⁺'s embedding around the original token embedding  
4. PVA score = `mean(loss(x⁺) + loss(x⁻)) / 2 − loss(x)` — positive score means the text sits at a local probability maximum, indicating memorisation

In [10]:
class SemanticParaphraser:
    """Symmetric paraphrase pairs via T5-base (Algorithm 2, Fu et al. 2024)."""
    T5_NAME = "t5-base"
    def __init__(self, mask_ratio=0.2, n_pairs=5):
        self.mask_ratio, self.n_pairs = mask_ratio, n_pairs
        print(f"  Loading T5 ({self.T5_NAME}) ...")
        self.t5_tok = AutoTokenizer.from_pretrained(self.T5_NAME)
        attn_kwargs = {"attn_implementation": "flash_attention_2"} if FLASH_ATTN else {}
        self.t5 = AutoModelForSeq2SeqLM.from_pretrained(
            self.T5_NAME, torch_dtype=DTYPE, **attn_kwargs).to(DEVICE).eval()
        with torch.no_grad():
            self.emb_matrix = self.t5.shared.weight.float().cpu()

    def _nearest(self, emb):
        return int((self.emb_matrix - emb.unsqueeze(0)).norm(dim=1).argmin().item())

    def paraphrase(self, text):
        ids = self.t5_tok.encode(text, truncation=True, max_length=256)
        if len(ids) < 4: return []
        pairs = []
        for _ in range(self.n_pairs):
            masked = list(ids)
            mi = [j for j in range(len(masked)) if random.random() < self.mask_ratio]
            if not mi: continue
            for j in mi: masked[j] = self.t5_tok.mask_token_id or 32099
            enc = self.t5_tok(self.t5_tok.decode(masked, skip_special_tokens=False),
                              return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
            with torch.no_grad():
                plus_ids = self.t5.generate(**enc, max_new_tokens=len(ids)+10)[0].cpu().tolist()
            plus_text = self.t5_tok.decode(plus_ids, skip_special_tokens=True)
            minus_ids = list(ids)
            for j in mi:
                eo = self.emb_matrix[ids[j]].float()
                ep = self.emb_matrix[plus_ids[j] if j < len(plus_ids) else ids[j]].float()
                minus_ids[j] = self._nearest(2.0 * eo - ep)
            pairs.append((plus_text, self.t5_tok.decode(minus_ids, skip_special_tokens=True)))
        return pairs

if USE_PVA:
    paraphraser = SemanticParaphraser(mask_ratio=PVA_MASK_RATIO, n_pairs=PVA_N_PAIRS)
    print("✓ PVA paraphraser ready")
else:
    paraphraser = None
    print("PVA disabled (USE_PVA=False) — PDC + grey-box only")

PVA disabled (USE_PVA=False) — PDC + grey-box only


## Step 9: Batched Scoring

**Three signals, one loop:**

| Signal | Formula | Intuition |
|---|---|---|
| PDC | `ref_loss − target_loss` | Fine-tuned model is more confident on its training data |
| Grey-box | Mean L2 norm, last hidden state | Fine-tuned models encode training data with higher-magnitude representations |
| PVA | `mean_neigh_loss − orig_loss` | Training samples sit at local probability maxima; neighbours have higher loss |

**Batching:** texts are processed in groups of 32. `CrossEntropyLoss(reduction='none')` recovers per-sample losses from a single batched forward pass, reducing GPU kernel launches from O(N) to O(N/32).

In [ ]:
# Step 9: Scoring — uses out.loss

def score_batch(model, tok, batch_texts, max_length=512, use_greybox=True):
    losses, hs_norms = [], []
    for text in batch_texts:
        enc = tok(text, return_tensors="pt", truncation=True,
                  max_length=max_length, padding=False).to(DEVICE)
        if enc["input_ids"].shape[1] < 2:
            losses.append(float("nan")); hs_norms.append(float("nan")); continue
        with torch.no_grad():
            out = model(**enc, labels=enc["input_ids"],
                        output_hidden_states=use_greybox)
        losses.append(out.loss.item())
        if use_greybox:
            am = enc["attention_mask"].float()
            hs = ((out.hidden_states[-1].norm(dim=-1) * am).sum(1)
                  / am.sum(1).clamp(min=1)).float().cpu().item()
            hs_norms.append(hs)
        else:
            hs_norms.append(float("nan"))
    return np.array(losses, dtype=float), np.array(hs_norms, dtype=float)

def get_loss_single(model, tok, text, max_length=512):
    enc = tok(text, return_tensors="pt", truncation=True,
               max_length=max_length, padding=False).to(DEVICE)
    if enc["input_ids"].shape[1] < 2: return float("nan")
    with torch.no_grad():
        return model(**enc, labels=enc["input_ids"]).loss.item()

def pva_score(text, model, tok, para, max_length=512):
    orig = get_loss_single(model, tok, text, max_length)
    if np.isnan(orig) or para is None: return float("nan")
    nbr = [(get_loss_single(model, tok, xp, max_length) +
            get_loss_single(model, tok, xm, max_length)) / 2
           for xp, xm in para.paraphrase(text)]
    nbr = [v for v in nbr if not np.isnan(v)]
    return float(np.mean(nbr) - orig) if nbr else float("nan")

def z_norm(a): return (a - a.mean()) / (a.std() + 1e-8)

# Load split 
print(f"Loading {DATASET_NAME} ({SCORE_SPLIT} split) ...")
ds      = load_dataset(DATASET_NAME, split=SCORE_SPLIT)
texts   = list(ds["text"])
all_ids = list(ds["id"])
labels  = None
if "is_member" in ds.column_names:
    raw = ds["is_member"]
    if any(v is not None for v in raw):
        labels = np.array(raw)
print(f"  {len(texts)} samples  |  labels={'available' if labels is not None else 'N/A'}")

# Scoring loop 
t_ls, r_ls, hs_ls, pva_ls, valid_idx = [], [], [], [], []

print(f"\nScoring {len(texts)} samples ...")
for start in tqdm(range(0, len(texts), SCORE_BATCH_SIZE)):
    end   = min(start + SCORE_BATCH_SIZE, len(texts))
    batch = texts[start:end]
    tl, th = score_batch(target_model, target_tok, batch, MAX_LENGTH, USE_GREYBOX)
    rl, _  = score_batch(ref_model,    ref_tok,    batch, MAX_LENGTH, False)
    for j in range(len(batch)):
        if np.isnan(tl[j]) or np.isnan(rl[j]): continue
        pva = pva_score(batch[j], target_model, target_tok, paraphraser, MAX_LENGTH) \
              if USE_PVA else 0.0
        if USE_PVA and np.isnan(pva): continue
        t_ls.append(tl[j]); r_ls.append(rl[j])
        hs_ls.append(th[j]); pva_ls.append(pva)
        valid_idx.append(start + j)

pdc_arr = np.array(r_ls) - np.array(t_ls)
hs_arr  = np.array(hs_ls)
pva_arr = np.array(pva_ls)
spv_arr = z_norm(pdc_arr)
if USE_GREYBOX and not np.all(np.isnan(hs_arr)):
    spv_arr = spv_arr + z_norm(hs_arr)
if USE_PVA:
    spv_arr = spv_arr + z_norm(pva_arr)
valid_ids = [all_ids[i] for i in valid_idx]

for fname, arr in [("pdc",pdc_arr),("hs",hs_arr),("pva",pva_arr),
                   ("spv",spv_arr),("ids",np.array(valid_ids))]:
    np.save(f"{CKPT_DIR}/{fname}.npy", arr)
print(f"\n\u2713 Scored {len(valid_idx)} / {len(texts)} samples  |  checkpointed \u2192 {CKPT_DIR}/")

Loading UBC-SLIME/colx_531_group_project (validation split) ...
  10000 samples  |  labels=available

Scoring 10000 samples ...


  0%|          | 0/313 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 7.59 GiB of which 26.62 MiB is free. Process 282873 has 5.67 GiB memory in use. Process 309925 has 170.00 MiB memory in use. Including non-PyTorch memory, this process has 1.26 GiB memory in use. Of the allocated memory 1.11 GiB is allocated by PyTorch, and 1.31 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Step 10: Evaluate — AUC & TPR@FPR=0.1

Only runs when labels are available (validation split). These are the numbers to put in your report table.

In [ ]:
if labels is not None:
    valid_labels = np.array([labels[i] for i in valid_idx])

    def report(name, scores, lbls):
        auc = roc_auc_score(lbls, scores)
        fpr, tpr, _ = roc_curve(lbls, scores)
        tpr01 = float(tpr[min(np.searchsorted(fpr, 0.1), len(tpr)-1)])
        print(f"  {name:<42}  AUC={auc:.4f}   TPR@FPR=0.1={tpr01:.4f}")
        return auc, tpr01

    print("="*65)
    print(f"  SPV-MIA Results  (split={SCORE_SPLIT})")
    print("="*65)
    report("Baseline: loss (135M vs 360M)",  np.zeros(len(valid_labels)), valid_labels)
    report("PDC only  (self-prompt LiRA)",   pdc_arr,                    valid_labels)
    if USE_GREYBOX and not np.all(np.isnan(hs_arr)):
        report("Grey-box  (hidden-state norm)",  z_norm(hs_arr),         valid_labels)
    if USE_PVA:
        report("PVA only  (prob. curvature)",    pva_arr,                valid_labels)
    report("SPV-MIA   (combined)",           spv_arr,                    valid_labels)
    print("="*65)
    print("\n→ Copy these numbers into Table 2 of Team_4.pdf")
else:
    print("Test split — no ground-truth labels. Skipping evaluation.")
    print("Run with SCORE_SPLIT='validation' first to get report numbers.")

  SPV-MIA Results  (split=validation)
  Baseline: loss (135M vs 360M)               AUC=0.5000   TPR@FPR=0.1=1.0000
  PDC only  (self-prompt LiRA)                AUC=0.7895   TPR@FPR=0.1=0.3576
  Grey-box  (hidden-state norm)               AUC=0.4927   TPR@FPR=0.1=0.0882
  SPV-MIA   (combined)                        AUC=0.6834   TPR@FPR=0.1=0.2722

→ Copy these numbers into Table 2 of Team_4.pdf


## Step 11: Save Submission CSV

Submits **raw float scores** (not binary 0/1) as `is_member`. Kaggle computes AUC from these directly.

In [ ]:
scored_df = pd.DataFrame({"id": valid_ids, "is_member": pdc_arr})
full_df       = pd.DataFrame({"id": all_ids})
submission_df = full_df.merge(scored_df, on="id", how="left").fillna(0.0)
submission_df["id"] = submission_df["id"].astype(int)

assert len(submission_df) == len(all_ids), \
    f"Row count mismatch: got {len(submission_df)}, expected {len(all_ids)}"
assert submission_df["id"].nunique() == len(all_ids), \
    f"Duplicate IDs detected — check the dataset."

submission_df[["id", "is_member"]].to_csv(OUTPUT_CSV, index=False)

pd.DataFrame({
    "id": valid_ids, "pdc": pdc_arr,
    "hs_norm": hs_arr, "pva": pva_arr, "spv": spv_arr,
}).to_csv(OUTPUT_CSV.replace(".csv", "_all_scores.csv"), index=False)

print(f"✓ Submission CSV → {OUTPUT_CSV}")
print(f"  Rows    : {len(submission_df)}  (all IDs, {len(submission_df)-len(valid_ids)} gaps filled with 0.0)")
print(f"  Columns : {list(submission_df[['id','is_member']].columns)}")
print(f"  Scores  : [{submission_df['is_member'].min():.4f}, {submission_df['is_member'].max():.4f}]")
print(f"\nUpload {OUTPUT_CSV} to Kaggle.")

✓ Submission CSV → submission_spv_mia.csv
  Rows    : 15000  (all IDs, 0 gaps filled with 0.0)
  Columns : ['id', 'is_member']
  Scores  : [-6.2719, 8.8110]

Upload submission_spv_mia.csv to Kaggle.


In [ ]:
scored_df     = pd.DataFrame({"id": valid_ids, "is_member": spv_arr})
full_df       = pd.DataFrame({"id": all_ids})
submission_df = full_df.merge(scored_df, on="id", how="left").fillna(0.0)
submission_df["id"] = submission_df["id"].astype(int)

assert len(submission_df) == 15000, \
    f"Row count mismatch: got {len(submission_df)}, expected 15000"
assert submission_df["id"].nunique() == 15000, \
    "Duplicate IDs detected — check the dataset."

submission_df[["id", "is_member"]].to_csv(OUTPUT_CSV, index=False)

print(f"✓ Submission CSV → {OUTPUT_CSV}")
print(f"  Rows    : {len(submission_df)}")
print(f"  Columns : {list(submission_df[['id','is_member']].columns)}")
print(f"  Scores  : [{submission_df['is_member'].min():.4f}, {submission_df['is_member'].max():.4f}]")
print(f"\nUpload {OUTPUT_CSV} to Kaggle.")

✓ Submission CSV → submission_spv_mia.csv
  Rows    : 15000
  Columns : ['id', 'is_member']
  Scores  : [-6.2719, 8.8110]

Upload submission_spv_mia.csv to Kaggle.


## Step 12 (Optional): Reload Checkpointed Scores

Run this if your Colab session disconnects after scoring. Skips Steps 5–9.

In [ ]:
pdc_arr   = np.load(f"{CKPT_DIR}/pdc.npy")
hs_arr    = np.load(f"{CKPT_DIR}/hs.npy")
pva_arr   = np.load(f"{CKPT_DIR}/pva.npy")
spv_arr   = np.load(f"{CKPT_DIR}/spv.npy")
valid_ids = np.load(f"{CKPT_DIR}/ids.npy").tolist()
print(f"✓ Reloaded {len(pdc_arr)} scored samples from {CKPT_DIR}/")

✓ Reloaded 15000 scored samples from ./spv_checkpoints/
